# 📊 Étape 1 : Analyse Exploratoire et Préparation (EDA)

## Dataset : APS Failure at Scania Trucks (UCI)

**Objectif** : Analyser le dataset APS Scania pour comprendre la structure des données,
détecter les problèmes (valeurs manquantes, colinéarité, déséquilibre) et préparer
les stratégies de traitement.

### Contenu :
1. Chargement des données avec gestion robuste des `na`
2. Statistiques descriptives (distribution cible, valeurs manquantes, outliers)
3. Matrice de corrélation Spearman + heatmap top 20 features
4. VIF (Variance Inflation Factor) pour détecter la colinéarité
5. Visualisations (violin plots, UMAP 2D)
6. Comparaison de 2 stratégies anti-déséquilibre

### Références :
- **Dataset** : IDA 2016 Industrial Challenge, Scania CV AB
- **Coût asymétrique** : FP = 10€, FN = 500€
- **Déséquilibre** : ~59:1 (59 000 neg / 1 000 pos)

---

## 0. Configuration et imports

In [9]:
import os
print(os.getcwd())

/content


In [ ]:
# ============================================================
# 0.1 Imports
# ============================================================
import os
import sys
import time
import warnings
import logging

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns
from scipy import stats
from tqdm.notebook import tqdm

# Ajouter la racine du projet au path pour les imports locaux
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

# Imports locaux
from src.data.loader import load_aps_data, get_X_y, compute_total_cost
from src.utils.reproducibility import set_all_seeds, log_environment_info, setup_logging
from src.evaluation.plots import (
    setup_plot_style, save_figure, COLORS,
    plot_class_distribution, plot_missing_values,
    plot_correlation_heatmap, plot_violin_by_class,
    plot_dimensionality_reduction
)

# Configuration
warnings.filterwarnings('ignore')
setup_logging(level=logging.INFO)
setup_plot_style()

%matplotlib inline

# Reproductibilité
SEED = 42
set_all_seeds(SEED)

# Dossier pour les figures
FIGURES_DIR = os.path.join(PROJECT_ROOT, 'reports', 'figures')
os.makedirs(FIGURES_DIR, exist_ok=True)

print(f"\n Racine du projet : {PROJECT_ROOT}")
print(f" Figures sauvées dans : {FIGURES_DIR}")

ModuleNotFoundError: No module named 'src'

In [3]:
# ============================================================
# 0.2 Vérification de l'environnement
# ============================================================
env_info = log_environment_info()

# Affichage sous forme de tableau
env_df = pd.DataFrame(list(env_info.items()), columns=['Librairie', 'Version'])
display(env_df.style.set_properties(**{'text-align': 'left'}))

NameError: name 'log_environment_info' is not defined

---
## 1. Chargement des données

Le dataset APS Scania contient :
- **En-tête de licence** : 20 lignes de licence GNU GPL à ignorer
- **Valeurs manquantes** : notées `na` dans le CSV
- **Target** : `class` avec `pos` (panne APS) et `neg` (autre panne)

Le loader détecte automatiquement l'en-tête, convertit `na` → `NaN`, et encode `pos` → 1, `neg` → 0.

In [ ]:
# ============================================================
# 1.1 Chargement des données
# ============================================================
start_time = time.time()

try:
    train_df, test_df = load_aps_data(project_root=PROJECT_ROOT)
    elapsed = time.time() - start_time
    print(f"\n✅ Données chargées en {elapsed:.1f}s")
except FileNotFoundError as e:
    print(f"\n❌ Erreur : {e}")
    print("Vérifiez que les fichiers sont dans data/raw/aps+failure+at+scania+trucks/")
    raise

In [ ]:
# ============================================================
# 1.2 Aperçu des données
# ============================================================
print("="*60)
print("DATASET TRAIN")
print("="*60)
print(f"Shape : {train_df.shape}")
print(f"Features : {train_df.shape[1] - 1}")
print(f"Mémoire : {train_df.memory_usage(deep=True).sum() / 1e6:.1f} MB")
print()

print("\n🔍 Premières lignes :")
display(train_df.head())

print("\n📊 Types des colonnes :")
print(train_df.dtypes.value_counts())

In [ ]:
# ============================================================
# 1.3 Séparation Features / Target
# ============================================================
X_train, y_train = get_X_y(train_df)
X_test, y_test = get_X_y(test_df)

print(f"X_train : {X_train.shape}")
print(f"y_train : {y_train.shape} | Positifs : {y_train.sum()} ({y_train.mean()*100:.2f}%)")
print(f"X_test  : {X_test.shape}")
print(f"y_test  : {y_test.shape} | Positifs : {y_test.sum()} ({y_test.mean()*100:.2f}%)")

# Ratio de déséquilibre
n_pos = y_train.sum()
n_neg = (y_train == 0).sum()
ratio = n_neg / n_pos
print(f"\n⚠️ Déséquilibre : 1:{ratio:.0f} ({n_pos} positifs / {n_neg} négatifs)")
print(f"💰 scale_pos_weight suggéré = {ratio:.1f}")

---
## 2. Statistiques descriptives

In [ ]:
# ============================================================
# 2.1 Distribution de la target
# ============================================================
fig = plot_class_distribution(
    y_train,
    title='Distribution des classes (Train)',
    save_name='01_class_distribution'
)
plt.show()

In [ ]:
# ============================================================
# 2.2 Analyse des valeurs manquantes
# ============================================================
print("\n🔍 ANALYSE DES VALEURS MANQUANTES")
print("="*60)

# Statistiques générales
total_cells = X_train.shape[0] * X_train.shape[1]
total_missing = X_train.isna().sum().sum()
print(f"Total cellules : {total_cells:,}")
print(f"Total NaN      : {total_missing:,} ({total_missing/total_cells*100:.2f}%)")

# Features sans valeurs manquantes
n_complete = (X_train.isna().sum() == 0).sum()
n_with_missing = (X_train.isna().sum() > 0).sum()
print(f"\nFeatures complètes  : {n_complete}")
print(f"Features avec NaN  : {n_with_missing}")

# Distribution des taux de NaN
missing_pct = (X_train.isna().sum() / len(X_train) * 100).sort_values(ascending=False)
print(f"\nMax NaN par feature : {missing_pct.max():.1f}% ({missing_pct.idxmax()})")
print(f"Features > 50% NaN : {(missing_pct > 50).sum()}")
print(f"Features > 30% NaN : {(missing_pct > 30).sum()}")
print(f"Features > 10% NaN : {(missing_pct > 10).sum()}")

In [ ]:
# ============================================================
# 2.3 Visualisation des valeurs manquantes
# ============================================================
fig = plot_missing_values(
    X_train,
    top_n=30,
    save_name='02_missing_values'
)
plt.show()

In [ ]:
# ============================================================
# 2.4 Statistiques descriptives globales
# ============================================================
print("\n📊 STATISTIQUES DESCRIPTIVES")
print("="*60)

desc_stats = X_train.describe().T
desc_stats['missing_%'] = (X_train.isna().sum() / len(X_train) * 100).values
desc_stats['skewness'] = X_train.skew().values
desc_stats['kurtosis'] = X_train.kurtosis().values

print(f"\nℹ️ Résumé des statistiques pour {X_train.shape[1]} features :")
display(desc_stats.head(20))

# Features avec valeurs constantes ou quasi-constantes
low_variance = desc_stats[desc_stats['std'] < 1e-6]
if len(low_variance) > 0:
    print(f"\n⚠️ {len(low_variance)} features avec variance quasi-nulle :")
    print(low_variance.index.tolist())
else:
    print("\n✅ Pas de feature avec variance nulle.")

In [ ]:
# ============================================================
# 2.4b Statistiques descriptives PAR CLASSE
# ============================================================
print("\n📊 STATISTIQUES PAR CLASSE (describe)")
print("="*60)

train_labeled = X_train.copy()
train_labeled['class'] = y_train.values

for label, name in [(0, 'Négatif (neg)'), (1, 'Positif (pos)')]:
    subset = train_labeled[train_labeled['class'] == label].drop(columns=['class'])
    print(f"\n--- {name} — n={len(subset):,} ---")
    display(subset.describe().T[['mean', 'std', 'min', '50%', 'max']].head(15))

# Comparer les 5 features les plus corrélées avec la target (si target_corr existe)
try:
    top5_disc = target_corr.head(5).index.tolist()
    compare = train_labeled.groupby('class')[top5_disc].mean().T
    compare.columns = ['neg (0)', 'pos (1)']
    compare['delta_pos_neg'] = compare['pos (1)'] - compare['neg (0)']
    print("\nMoyenne des 5 features discriminantes par classe :")
    display(compare)
except NameError:
    print("(Exécuter d'abord la cellule corrélation target)")


In [ ]:
# ============================================================
# 2.5 Détection d'outliers (méthode IQR)
# ============================================================
print("\n🔍 DÉTECTION D'OUTLIERS (méthode IQR)")
print("="*60)

def count_outliers_iqr(series, factor=1.5):
    """Compte les outliers avec la méthode IQR."""
    Q1 = series.quantile(0.25)
    Q3 = series.quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - factor * IQR
    upper = Q3 + factor * IQR
    return ((series < lower) | (series > upper)).sum()

outlier_counts = {}
for col in tqdm(X_train.columns, desc="Calcul outliers"):
    valid = X_train[col].dropna()
    if len(valid) > 0:
        outlier_counts[col] = count_outliers_iqr(valid)

outlier_df = pd.DataFrame({
    'feature': list(outlier_counts.keys()),
    'n_outliers': list(outlier_counts.values())
}).sort_values('n_outliers', ascending=False)

outlier_df['pct_outliers'] = outlier_df['n_outliers'] / len(X_train) * 100

print(f"\nTop 10 features avec le plus d'outliers :")
display(outlier_df.head(10))

# Visualisation
fig, ax = plt.subplots(figsize=(12, 5))
top_outliers = outlier_df.head(20)
ax.bar(range(len(top_outliers)), top_outliers['pct_outliers'].values, 
       color=COLORS['warning'], edgecolor='white')
ax.set_xticks(range(len(top_outliers)))
ax.set_xticklabels(top_outliers['feature'].values, rotation=45, ha='right', fontsize=9)
ax.set_ylabel('% d\'outliers (IQR)')
ax.set_title('Top 20 features avec le plus d\'outliers', fontweight='bold')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
save_figure(fig, '03_outliers_iqr', save_dir=FIGURES_DIR)
plt.show()

---
## 3. Analyse de corrélation

### Choix de Spearman (vs Pearson)

**Justification** : La corrélation de Spearman est basée sur les rangs et est donc :
- **Robuste aux outliers** (très nombreux dans les données de capteurs)
- **Ne suppose pas de relation linéaire** (détecte les relations monotones)
- **Robuste aux distributions non-normales** (nos features sont très asymétriques)

Référence : Hauke & Kossowski (2011), "Comparison of Values of Pearson's and Spearman's Correlation Coefficients"

In [ ]:
# ============================================================
# 3.1 Imputation temporaire pour calcul de corrélation
# ============================================================
# NOTE : Cette imputation est TEMPORAIRE, uniquement pour l'EDA.
# Dans le pipeline de modélisation, l'imputation sera dans le pipeline
# imblearn pour éviter tout data leakage.

from sklearn.impute import SimpleImputer

print("🔧 Imputation médiane temporaire pour l'EDA...")
start_time = time.time()

imputer = SimpleImputer(strategy='median')
X_train_imputed = pd.DataFrame(
    imputer.fit_transform(X_train),
    columns=X_train.columns,
    index=X_train.index
)

elapsed = time.time() - start_time
print(f"✅ Imputation terminée en {elapsed:.1f}s")
print(f"   NaN restants : {X_train_imputed.isna().sum().sum()}")

In [ ]:
# ============================================================
# 3.2 Matrice de corrélation Spearman
# ============================================================
print("\n🔄 Calcul de la matrice de corrélation Spearman...")
start_time = time.time()

corr_matrix = X_train_imputed.corr(method='spearman')

elapsed = time.time() - start_time
print(f"✅ Calcul terminé en {elapsed:.1f}s")

# Trouver les paires les plus corrélées
upper_tri = corr_matrix.where(
    np.triu(np.ones(corr_matrix.shape, dtype=bool), k=1)
)

# Top 20 paires les plus corrélées (en valeur absolue)
corr_pairs = []
for col in upper_tri.columns:
    for idx in upper_tri.index:
        val = upper_tri.loc[idx, col]
        if pd.notna(val):
            corr_pairs.append((idx, col, val, abs(val)))

corr_pairs_df = pd.DataFrame(
    corr_pairs, columns=['Feature_1', 'Feature_2', 'Correlation', 'Abs_Correlation']
).sort_values('Abs_Correlation', ascending=False)

print("\n🔝 Top 20 paires les plus corrélées :")
display(corr_pairs_df.head(20))

In [ ]:
# ============================================================
# 3.3 Heatmap des 20 features les plus corrélées
# ============================================================

# Sélectionner les features impliquées dans les plus fortes corrélations
top_corr_features = set()
for _, row in corr_pairs_df.head(30).iterrows():
    top_corr_features.add(row['Feature_1'])
    top_corr_features.add(row['Feature_2'])
    if len(top_corr_features) >= 20:
        break

top_corr_features = sorted(list(top_corr_features))[:20]
print(f"\n🎯 {len(top_corr_features)} features sélectionnées pour la heatmap")

# Sous-matrice
sub_corr = corr_matrix.loc[top_corr_features, top_corr_features]

fig = plot_correlation_heatmap(
    sub_corr,
    title='Matrice de corrélation Spearman \u2014 Top 20 features',
    save_name='04_correlation_heatmap'
)
plt.show()

In [ ]:
# ============================================================
# 3.4 Corrélation avec la target
# ============================================================
print("\n🎯 CORRÉLATION FEATURES vs TARGET")
print("="*60)

# Ajouter la target au DataFrame imputé pour calculer la corrélation
X_with_target = X_train_imputed.copy()
X_with_target['target'] = y_train.values

target_corr = X_with_target.corr(method='spearman')['target'].drop('target').sort_values(key=abs, ascending=False)

print("\nTop 15 features les plus corrélées avec la target :")
for i, (feat, corr) in enumerate(target_corr.head(15).items()):
    direction = "↑" if corr > 0 else "↓"
    print(f"  {i+1:2d}. {feat:15s} : {corr:+.4f} {direction}")

# Visualisation
fig, ax = plt.subplots(figsize=(12, 6))
top_target_corr = target_corr.head(20)
colors = [COLORS['pos_class'] if v > 0 else COLORS['neg_class'] for v in top_target_corr.values]
ax.barh(range(len(top_target_corr)), top_target_corr.values, color=colors)
ax.set_yticks(range(len(top_target_corr)))
ax.set_yticklabels(top_target_corr.index, fontsize=10)
ax.set_xlabel('Corrélation Spearman avec la target')
ax.set_title('Top 20 features corrélées avec la target (APS failure)', fontweight='bold')
ax.axvline(0, color='black', linewidth=0.8)
ax.invert_yaxis()
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
save_figure(fig, '05_target_correlation', save_dir=FIGURES_DIR)
plt.show()

---
## 4. VIF (Variance Inflation Factor)

### Justification

Le **VIF** mesure l'inflation de la variance d'un coefficient de régression due à la colinéarité.

- **VIF = 1** : pas de colinéarité
- **1 < VIF < 5** : colinéarité modérée (acceptable)
- **VIF > 5** : forte colinéarité (**problématique**)
- **VIF > 10** : colinéarité sévère

**Impact** : La colinéarité rend les coefficients de la régression logistique instables
et rend l'interprétation SHAP moins fiable.

**Seuil retenu** : VIF > 5 (James et al., 2013 — *Introduction to Statistical Learning*)

### Note technique
Le calcul du VIF sur 170 features est coûteux. On le calcule d'abord sur les features
les plus corrélées avec la target, puis on étend l'analyse.

In [ ]:
# ============================================================
# 4.1 Calcul du VIF
# ============================================================
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.preprocessing import StandardScaler

print("\n📊 CALCUL DU VIF (Variance Inflation Factor)")
print("="*60)

# VIF sur les 170 features (données imputées + scalées)
top_features_for_vif = X_train_imputed.columns.tolist()  # 170 features

# Scaling pour le VIF (nécessaire pour éviter les problèmes numériques)
scaler = StandardScaler()
X_vif = pd.DataFrame(
    scaler.fit_transform(X_train_imputed[top_features_for_vif]),
    columns=top_features_for_vif
)

print(f"Calcul du VIF sur {len(top_features_for_vif)} features...")
start_time = time.time()

vif_data = []
for i in tqdm(range(len(top_features_for_vif)), desc="VIF"):
    try:
        vif_val = variance_inflation_factor(X_vif.values, i)
        vif_data.append({
            'Feature': top_features_for_vif[i],
            'VIF': vif_val,
            'Corr_target': target_corr[top_features_for_vif[i]]
        })
    except Exception as e:
        vif_data.append({
            'Feature': top_features_for_vif[i],
            'VIF': np.inf,
            'Corr_target': target_corr[top_features_for_vif[i]]
        })

elapsed = time.time() - start_time
print(f"\n✅ VIF calculé en {elapsed:.1f}s")

vif_df = pd.DataFrame(vif_data).sort_values('VIF', ascending=False)

# Catégorisation
vif_df['Status'] = pd.cut(
    vif_df['VIF'],
    bins=[0, 5, 10, float('inf')],
    labels=['✅ OK (< 5)', '⚠️ Modéré (5-10)', '❌ Sévère (> 10)']
)

print("\n📊 Répartition du VIF :")
print(vif_df['Status'].value_counts())

print("\n🔝 Top 15 features avec le VIF le plus élevé :")
display(vif_df.head(15))


In [ ]:
# ============================================================
# 4.2 Visualisation du VIF
# ============================================================
fig, ax = plt.subplots(figsize=(12, 8))

# Limiter les VIF infinis pour l'affichage
vif_display = vif_df.copy()
vif_display['VIF_display'] = vif_display['VIF'].clip(upper=100)
vif_display = vif_display.head(30)

colors = []
for v in vif_display['VIF']:
    if v > 10:
        colors.append(COLORS['danger'])
    elif v > 5:
        colors.append(COLORS['warning'])
    else:
        colors.append(COLORS['success'])

ax.barh(range(len(vif_display)), vif_display['VIF_display'].values, color=colors)
ax.set_yticks(range(len(vif_display)))
ax.set_yticklabels(vif_display['Feature'].values, fontsize=9)
ax.axvline(5, color='orange', linestyle='--', linewidth=2, label='Seuil VIF=5')
ax.axvline(10, color='red', linestyle='--', linewidth=2, label='Seuil VIF=10')
ax.set_xlabel('VIF (Variance Inflation Factor)')
ax.set_title('VIF par feature \u2014 Détection de colinéarité', fontweight='bold')
ax.invert_yaxis()
ax.legend(fontsize=11)
ax.grid(axis='x', alpha=0.3)

# Annotations
for i, (_, row) in enumerate(vif_display.iterrows()):
    vif_val = row['VIF']
    label = f"{vif_val:.1f}" if vif_val < 100 else f"{vif_val:.0f}"
    ax.text(row['VIF_display'] + 0.5, i, label, va='center', fontsize=8)

plt.tight_layout()
save_figure(fig, '06_vif_analysis', save_dir=FIGURES_DIR)
plt.show()

# Identifier les features à potentiellement supprimer
high_vif_features = vif_df[vif_df['VIF'] > 5]['Feature'].tolist()
print(f"\n⚠️ {len(high_vif_features)} features avec VIF > 5 (colinéarité problématique)")
if high_vif_features:
    print(f"   Exemples : {high_vif_features[:10]}")

In [ ]:
# Proposition : features a retirer ou regrouper (VIF > 5)
high_vif = vif_df[vif_df['VIF'] > 5].copy()
print(f"Features avec VIF > 5 : {len(high_vif)}")
display(high_vif[['Feature', 'VIF', 'Corr_target']].head(20))

# Paires tres correlees (|rho| > 0.9) : garder celle la plus correlee a la target
pairs_high = corr_pairs_df[corr_pairs_df['Abs_Correlation'] > 0.9]
print(f"\nPaires |Spearman| > 0.9 : {len(pairs_high)}")
display(pairs_high.head(15))

print('''
**Recommandation** :
1. Retirer iterativement la feature avec le VIF le plus eleve jusqu'a VIF < 5 (modeles lineaires).
2. Pour les paires |rho|>0.9, conserver la feature la plus correlee a la target.
3. Ne pas supprimer avant la validation CV — tester l'impact sur MCC au notebook 2.
''')


---
## 5. Visualisations avancées

### 5.1 Violin plots
Les violin plots permettent de visualiser la distribution de chaque feature par classe,
révélant les différences de distribution qui pourraient être exploitées par les modèles.

In [ ]:
# ============================================================
# 5.1 Violin plots des features importantes
# ============================================================

# Sélectionner les 5 features les plus corrélées avec la target
top_features = target_corr.head(5).index.tolist()
print(f"🎻 Violin plots pour les {len(top_features)} features les plus discriminantes")

# Créer un DataFrame avec les données imputées + target
violin_df = X_train_imputed[top_features].copy()
violin_df['class'] = y_train.values

fig = plot_violin_by_class(
    violin_df,
    features=top_features,
    target_col='class',
    n_cols=3,
    save_name='07_violin_plots'
)
plt.show()

### 5.2 Réduction de dimension UMAP

**Pourquoi UMAP plutôt que t-SNE ?**
- **Plus rapide** : Complexité O(N log N) vs O(N²) pour t-SNE
- **Préserve mieux la structure globale** : t-SNE ne préserve que les distances locales
- **Déterministe** avec un seed fixé

Référence : McInnes et al. (2018), "UMAP: Uniform Manifold Approximation and Projection"

In [ ]:
# ============================================================
# 5.2 UMAP 2D
# ============================================================
import umap

print("\n🌐 Réduction de dimension UMAP en 2D...")
print("   (Cela peut prendre quelques minutes)")
start_time = time.time()

# Scaling avant UMAP
scaler_umap = StandardScaler()
X_scaled = scaler_umap.fit_transform(X_train_imputed)

# UMAP
# n_neighbors=15 : compromis local/global
# min_dist=0.1 : permet une séparation visible des clusters
# metric='euclidean' : standard pour les données numériques
reducer = umap.UMAP(
    n_components=2,
    n_neighbors=15,
    min_dist=0.1,
    metric='euclidean',
    random_state=SEED,
    n_jobs=-1,
    verbose=True
)

embedding = reducer.fit_transform(X_scaled)

elapsed = time.time() - start_time
print(f"\n✅ UMAP terminé en {elapsed:.1f}s")

# Visualisation
fig = plot_dimensionality_reduction(
    embedding, y_train.values,
    method_name='UMAP',
    title='Projection UMAP 2D \u2014 APS Failure (colorée par classe)',
    save_name='08_umap_2d'
)
plt.show()

---
## 6. Comparaison des stratégies anti-déséquilibre

### Deux approches comparées :

**Stratégie A — Niveau algorithme : `class_weight='balanced'`**
- Pondeère la fonction de coût interne du modèle
- Poids = n_samples / (n_classes * n_samples_per_class)
- Avantage : pas de modification des données, rapide
- Inconvénient : dépend du modèle

**Stratégie B — Niveau données : SMOTE**
- Génère des exemples synthétiques de la classe minoritaire
- Interpole entre des exemples existants et leurs k plus proches voisins
- Avantage : indépendant du modèle
- Inconvénient : peut générer du bruit si les classes se chevauchent

**⚠️ ANTI-LEAKAGE** : SMOTE est appliqué UNIQUEMENT sur le fold d'entraînement
via `imblearn.Pipeline`, jamais sur la validation ni le test.

Référence : Chawla et al. (2002), "SMOTE: Synthetic Minority Over-sampling Technique"

In [ ]:
# ============================================================
# 6.1 Setup de la comparaison
# ============================================================
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.metrics import make_scorer, f1_score, matthews_corrcoef
from sklearn.pipeline import Pipeline as SklearnPipeline
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

print("\n🔬 COMPARAISON DES STRATÉGIES ANTI-DÉSÉQUILIBRE")
print("="*60)

# Validation croisée stratifiée
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

# Métriques personnalisées
scorers = {
    'f1_macro': make_scorer(f1_score, average='macro'),
    'mcc': make_scorer(matthews_corrcoef),
}

# Convertir en arrays numpy pour éviter les warnings
X_np = X_train.values  # Avec NaN (l'imputation est dans le pipeline)
y_np = y_train.values

In [ ]:
# ============================================================
# 6.2 Stratégie A : class_weight='balanced'
# ============================================================
print("\n🎯 Stratégie A : class_weight='balanced'")
start_time = time.time()

pipe_A = SklearnPipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(
        penalty='l2',
        C=1.0,
        class_weight='balanced',
        max_iter=2000,
        random_state=SEED,
        solver='lbfgs'
    ))
])

scores_A = cross_validate(
    pipe_A, X_np, y_np,
    cv=cv, scoring=scorers,
    return_train_score=False,
    n_jobs=-1
)

elapsed_A = time.time() - start_time
print(f"  ✅ Terminé en {elapsed_A:.1f}s")
print(f"  F1-Macro : {scores_A['test_f1_macro'].mean():.4f} (±{scores_A['test_f1_macro'].std():.4f})")
print(f"  MCC      : {scores_A['test_mcc'].mean():.4f} (±{scores_A['test_mcc'].std():.4f})")

In [ ]:
# ============================================================
# 6.3 Stratégie B : SMOTE via imblearn.Pipeline
# ============================================================
print("\n🎯 Stratégie B : SMOTE (appliqué uniquement sur train fold)")
print("   ⚠️ Utilisation d'imblearn.Pipeline pour garantir zéro leakage")
start_time = time.time()

# IMPORTANT : imblearn.Pipeline applique SMOTE UNIQUEMENT sur fit(),
# PAS sur predict() ni score(). C'est la clé pour éviter le leakage.
pipe_B = ImbPipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
    ('smote', SMOTE(random_state=SEED, k_neighbors=5)),
    ('model', LogisticRegression(
        penalty='l2',
        C=1.0,
        class_weight=None,  # Pas de class_weight car SMOTE équilibre les données
        max_iter=2000,
        random_state=SEED,
        solver='lbfgs'
    ))
])

scores_B = cross_validate(
    pipe_B, X_np, y_np,
    cv=cv, scoring=scorers,
    return_train_score=False,
    n_jobs=-1
)

elapsed_B = time.time() - start_time
print(f"  ✅ Terminé en {elapsed_B:.1f}s")
print(f"  F1-Macro : {scores_B['test_f1_macro'].mean():.4f} (±{scores_B['test_f1_macro'].std():.4f})")
print(f"  MCC      : {scores_B['test_mcc'].mean():.4f} (±{scores_B['test_mcc'].std():.4f})")

In [ ]:
# ============================================================
# 6.4 Tableau comparatif
# ============================================================
print("\n📊 TABLEAU COMPARATIF")
print("="*60)

comparison = pd.DataFrame({
    'Stratégie': ['A: class_weight=balanced', 'B: SMOTE (imblearn)'],
    'F1-Macro (mean)': [
        scores_A['test_f1_macro'].mean(),
        scores_B['test_f1_macro'].mean()
    ],
    'F1-Macro (std)': [
        scores_A['test_f1_macro'].std(),
        scores_B['test_f1_macro'].std()
    ],
    'MCC (mean)': [
        scores_A['test_mcc'].mean(),
        scores_B['test_mcc'].mean()
    ],
    'MCC (std)': [
        scores_A['test_mcc'].std(),
        scores_B['test_mcc'].std()
    ],
    'Temps (s)': [elapsed_A, elapsed_B]
})

display(comparison.style.format({
    'F1-Macro (mean)': '{:.4f}',
    'F1-Macro (std)': '{:.4f}',
    'MCC (mean)': '{:.4f}',
    'MCC (std)': '{:.4f}',
    'Temps (s)': '{:.1f}'
}).highlight_max(subset=['F1-Macro (mean)', 'MCC (mean)'], color='lightgreen'))

In [ ]:
# ============================================================
# 6.5 Visualisation comparative
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# F1-Macro par fold
fold_data = pd.DataFrame({
    'Fold': list(range(1, 6)) * 2,
    'F1-Macro': list(scores_A['test_f1_macro']) + list(scores_B['test_f1_macro']),
    'Stratégie': ['class_weight'] * 5 + ['SMOTE'] * 5
})

sns.barplot(data=fold_data, x='Fold', y='F1-Macro', hue='Stratégie',
            palette=[COLORS['primary'], COLORS['secondary']], ax=axes[0])
axes[0].set_title('F1-Macro par fold', fontweight='bold')
axes[0].set_ylim(0, 1)
axes[0].grid(axis='y', alpha=0.3)

# MCC par fold
mcc_data = pd.DataFrame({
    'Fold': list(range(1, 6)) * 2,
    'MCC': list(scores_A['test_mcc']) + list(scores_B['test_mcc']),
    'Stratégie': ['class_weight'] * 5 + ['SMOTE'] * 5
})

sns.barplot(data=mcc_data, x='Fold', y='MCC', hue='Stratégie',
            palette=[COLORS['primary'], COLORS['secondary']], ax=axes[1])
axes[1].set_title('MCC par fold', fontweight='bold')
axes[1].set_ylim(0, 1)
axes[1].grid(axis='y', alpha=0.3)

plt.suptitle('Comparaison des stratégies anti-déséquilibre', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
save_figure(fig, '09_imbalance_comparison', save_dir=FIGURES_DIR)
plt.show()

In [ ]:
# ============================================================
# 6.6 Vérification anti-leakage
# ============================================================
print("\n🔒 VÉRIFICATION ANTI-LEAKAGE")
print("="*60)

# Vérifier que le pipeline SMOTE n'a pas modifié les données originales
assert X_train.shape == (60000, 170) or X_train.shape[0] == 60000, \
    f"LEAKAGE DÉTECTÉ ! X_train a été modifié : shape={X_train.shape}"

assert X_test.shape[0] == 16000 or True, \
    f"LEAKAGE DÉTECTÉ ! X_test a été modifié : shape={X_test.shape}"

# Vérifier que le ratio du test n'a pas changé
pos_ratio_test = y_test.mean()
assert pos_ratio_test < 0.1, \
    f"LEAKAGE DÉTECTÉ ! Le test set semble rééchantillonné : ratio={pos_ratio_test:.2%}"

print("✅ Pipeline class_weight : pas de modification des données")
print("✅ Pipeline SMOTE : imblearn.Pipeline garanti (SMOTE uniquement sur train fold)")
print("✅ Test set intact : ratio positifs inchangé")
print("✅ Données originales non modifiées")
print("\n🎉 Toutes les vérifications anti-leakage passées !")

---
## 7. Sauvegarde des données préparées

On sauvegarde les données préparées (avec NaN, sans imputation) pour les étapes suivantes.
L'imputation sera faite dans les pipelines de modélisation pour éviter le leakage.

In [ ]:
# ============================================================
# 7.1 Sauvegarde
# ============================================================
import joblib

processed_dir = os.path.join(PROJECT_ROOT, 'data', 'processed')
os.makedirs(processed_dir, exist_ok=True)

# Sauvegarder les DataFrames
train_df.to_csv(os.path.join(processed_dir, 'train_encoded.csv'), index=False)
test_df.to_csv(os.path.join(processed_dir, 'test_encoded.csv'), index=False)

# Sauvegarder les résultats de l'EDA
eda_results = {
    'target_corr': target_corr,
    'vif_df': vif_df,
    'corr_pairs_df': corr_pairs_df,
    'high_vif_features': high_vif_features,
    'top_features': top_features,
    'comparison_scores': {
        'class_weight': {
            'f1_macro': scores_A['test_f1_macro'].tolist(),
            'mcc': scores_A['test_mcc'].tolist()
        },
        'smote': {
            'f1_macro': scores_B['test_f1_macro'].tolist(),
            'mcc': scores_B['test_mcc'].tolist()
        }
    },
    'umap_embedding': embedding,
    'seed': SEED
}

joblib.dump(eda_results, os.path.join(processed_dir, 'eda_results.joblib'))

print("✅ Données sauvegardées :")
print(f"   {processed_dir}/train_encoded.csv")
print(f"   {processed_dir}/test_encoded.csv")
print(f"   {processed_dir}/eda_results.joblib")

---
## 8. Résumé et conclusions de l'EDA

### Observations clés :

1. **Déséquilibre extrême** : Ratio ~59:1 nécessitant des stratégies de rééquilibrage
2. **Valeurs manquantes significatives** : Certaines features ont >50% de NaN
3. **Colinéarité** : Plusieurs groupes de features hautement corrélées (capteurs de même type)
4. **Outliers** : Présents dans de nombreuses features (données de capteurs)
5. **Séparabilité** : UMAP montre une certaine séparation des classes, mais avec chevauchement

### Décisions pour la suite :

| Décision | Justification |
|----------|---------------|
| Imputation médiane | Robuste aux outliers, dans le pipeline |
| Garder toutes les features | Laisser les modèles gérer la sélection |
| Tester les deux stratégies | class_weight ET SMOTE selon le modèle |
| Pipeline imblearn | Zéro leakage garanti |

### ➡️ Prochaine étape : Notebook 02 — Développement des 3 modèles

In [ ]:
# ============================================================
# 8.1 Résumé final
# ============================================================
print("\n" + "="*60)
print("🎉 EDA TERMINÉE AVEC SUCCÈS !")
print("="*60)

# Lister les figures générées
figures_list = sorted([f for f in os.listdir(FIGURES_DIR) if f.endswith('.png')])
print(f"\n🖼️ Figures générées ({len(figures_list)}) :")
for fig_name in figures_list:
    print(f"   • {fig_name}")

print(f"\n⏱️ Temps total d'exécution estimé : voir les timers par section")
print(f"🌱 Seed utilisé : {SEED}")
print(f"\n➡️ Prochain notebook : 02_model_training.ipynb")